# NanoCoder: 1 - Source Data

*Choose the training sources, then shape them into one clean, correctly-proportioned pretraining corpus.*

A language model can only learn what its training data shows it. So before we design a tokenizer or a single layer of the network, we decide **what the model reads** - that choice, more than any architectural trick, sets the ceiling on what the finished model can do. This notebook builds that corpus, end to end.

## In this notebook, we will:

- Assemble an *interleaved* Hugging Face pretraining dataset from **7 sources**, each chosen to specialize the model in technical explanation and Python code
- Normalize every source down to a single `text` field, so sources with different schemas can be mixed
- Solve the "documents vs. tokens" trap - setting the mix we want measured in *tokens*, not documents
- Size the corpus against a token budget, so the model sees enough to learn but not so much that it memorizes
- Push the finished dataset to the Hugging Face Hub

## Learning Goals

By the end, you should understand:

- **Why source selection is itself a modeling decision** - every dataset maps to a capability we want the model to have
- **Why a document mix is not a token mix**, and how to solve for the sampling weights that hit a target token proportion
- **How a tokens-per-parameter budget** (the Chinchilla heuristic) decides how much data to compile
- **Why cheap streaming filters** beat exact-but-slow ones at this scale, and what that trade costs you

The finished datasets, tokenizer, and models all live on [HuggingFace](https://huggingface.co/torq1); the full codebase is on [GitHub](https://github.com/tannerorourke/NanoCoder). These notebooks are meant to be read alongside the repository - fork it and follow along.

### Notebook jumplinks

- [NanoCoder: Introduction](https://colab.research.google.com/drive/1nwCKzG2hFeQyrmINdJxji_f_vJVf9AUv?usp=sharing)
- [NanoCoder 2: Tokenizer](https://colab.research.google.com/drive/1jKz9KB1qISMOmQn7xffbHVzP4WiNOXcu?usp=sharing)
- [NanoCoder 3: Base Model](https://colab.research.google.com/drive/1XE-3OBhWYlssQRXc2_s9lZnhmjfWqCVB?usp=sharing)
- [NanoCoder 4: SFT](https://colab.research.google.com/drive/1TTdyEzp106U--42HO6PQ2WsCjJJbZY96?usp=sharing)
- [NanoCoder 5: RFT & DPO](https://colab.research.google.com/drive/1urlRWYFI1ojhvB64t46Epv4g3zybZSgv?usp=sharing)

In [ ]:
%pip install -q git+https://github.com/tannerorourke/NanoCoder.git
# -- Editing the source as you read? Clone and install editable instead:
#      !git clone https://github.com/<you>/NanoCoder.git
#      %pip install -q -e NanoCoder

import nanocoder
print(f"nanocoder {nanocoder.__version__} ready")

## Part 1: From the goal to the data

The data a model trains on approximates what the model learns - full stop. A striking example: GPT-3.5-turbo-instruct, built to continue English text, turned out to [play chess at roughly the level of a skilled human](https://dynomight.substack.com/p/more-chess), despite being older and much smaller than today's models. Nobody trained it on chess on purpose. It learned because chess games written in PGN notation happened to sit in its training data, and predicting the next move forced it to internalize the rules and track the board state.

The lesson to carry through this notebook: the algorithms behind these models are unreasonably good at pulling structure out of whatever data you show them - the patterns you intended, and plenty you didn't. The corpus is where we steer the model, so we choose it deliberately.

### Finding the data from the goal

Our goal reduces to a single sentence: **take a technical instruction, produce Python.** Each source we add exists to cover one capability that goal demands:

1. **Fluency in real Python** - the backbone of the corpus. Imports, class layout, error handling, whole files rather than snippets.
2. **Technical language** - the concepts that get explained and then coded. A model that can write binary search *for a described problem* is useful; one that only reproduces the textbook version is not.
3. **The prompt → code interface** - the model has to answer a request, not just continue a file. This is instruction data, and it is by far the scarcest ingredient we need.

Every source below maps to one of these three jobs. That mapping is what gives the corpus its structure.

## Part 2: The seven sources

We group the seven sources by the job each one does.

### Code backbone

[codeparrot-clean](https://huggingface.co/datasets/codeparrot/codeparrot-clean) is our **Python backbone**. It is the only source that ships whole `.py` files, and it carries roughly 3x the token mass of every instruction set combined. Whole files are what teach the model *file shape* - how real code is laid out, not just how a single function reads.

### Instruction sets

Instruction data - a request paired with its answer - is the scarcest thing we need, so we pool five sets rather than lean on one and inherit its quirks:

- [glaive-code-assistant](https://huggingface.co/datasets/glaiveai/glaive-code-assistant)
- [TinyCodes](https://huggingface.co/datasets/nampdn-ai/tiny-codes)
- [Magicoder-Evol-Instruct-110K](https://huggingface.co/datasets/ise-uiuc/Magicoder-Evol-Instruct-110K)
- [Magicoder-OSS-Instruct-75K](https://huggingface.co/datasets/ise-uiuc/Magicoder-OSS-Instruct-75K) (seeded from real open-source snippets rather than from Evol-Instruct, so it fails differently than the Evol sets do - pooling them cancels some of each set's bias)
- [CodeFeedback-Filtered](https://huggingface.co/datasets/m-a-p/CodeFeedback-Filtered-Instruction)

Together these are only ~222M tokens, which is exactly why they get sampled hardest - more on epoch counts in Part 5.

### Technical prose

The model also has to *understand* the questions it is asked - the "explain the concept, then write it" prompts. For that we use [Cosmopedia v2](https://huggingface.co/datasets/HuggingFaceTB/smollm-corpus), long-form technical explanations. This kind of data is effectively unlimited and always somewhat useful, which makes it the easiest source to accidentally over-weight - so we cap it hard (Part 3).

| Source | HF dataset | Role in the model |
|---|---|---|
| `codeparrot` | `codeparrot/codeparrot-clean` | Raw `.py` files - the **only** source that teaches *file shape* |
| `glaive` | `glaiveai/glaive-code-assistant` | `## Task / ## Solution` - the prompt → code interface |
| `tinycodes` | `nampdn-ai/tiny-codes` | instruct pairs (Python-filtered) |
| `magicoder_evol` | `ise-uiuc/Magicoder-Evol-Instruct-110K` | instruct pairs |
| `codefeedback` | `m-a-p/CodeFeedback-Filtered-Instruction` | instruct pairs |
| `magicoder_oss` | `ise-uiuc/Magicoder-OSS-Instruct-75K` | instruct pairs |
| `cosmo` | `HuggingFaceTB/smollm-corpus` (cosmopedia-v2) | technical **prose** - for questions whose answer isn't a function |

In [ ]:
# --- Peek at the raw sources before any filtering. Streaming, so nothing downloads in full.
from datasets import load_dataset

# (repo, config, the column the payload actually lives in)
peek = {
    "codeparrot": ("codeparrot/codeparrot-clean", None, "content"),
    "glaive":     ("glaiveai/glaive-code-assistant", None, "answer"),
    "cosmo":      ("HuggingFaceTB/smollm-corpus", "cosmopedia-v2", "text"),
}

for name, (repo, sub, col) in peek.items():
    ex = next(iter(load_dataset(repo, sub, split="train", streaming=True)))
    body = ex[col]
    print(f"===== {name}  columns={list(ex)}")
    print(f"      {col}: {len(body):,} chars, {sum(ord(c) > 127 for c in body)} non-ascii")
    print(body[:300].rstrip(), "\n...\n")

## Part 3: The document-vs-token trap

Here is the first place intuition quietly fails. Documents are not all the same length: a raw `.py` file dwarfs a short instruction answer. If we interleaved the sources *per document* - "pick 30% of documents from source A" - the longer-document sources would contribute far more *tokens* than their document share suggests, because the model trains on tokens, not documents.

A single global length cap does not fix this. Trimming every document to the same length silently favors whichever source is naturally shortest, and it guts the long-form prose source we added for its length. Caps are thus applied **per source** instead, tuned to each source's own length distribution.

In [ ]:
import statistics
from itertools import islice

def median_chars(repo, sub, col, n=300):
    stream = load_dataset(repo, sub, split="train", streaming=True)
    return statistics.median(len(ex[col]) for ex in islice(stream, n))

for name, (repo, sub, col) in peek.items():
    print(f"{name:<12}{median_chars(repo, sub, col):>8,.0f} median chars")

# -- Measured across all seven: cosmo 3,676 / codeparrot 4,512 / tinycodes 1,758 /
#    magicoder response 1,467 / glaive 1,238. Roughly a 4x spread.

# Mean length of a doc that SURVIVES the caps and the language filter, in tokens. 
# Part 4 solves the sampling weights from these.
mean_tokens = {
    "codeparrot": 1100, "cosmo": 900, "glaive": 210, "tinycodes": 170,
    "magicoder_evol": 240, "codefeedback": 230, "magicoder_oss": 260,
}

| source | role | pool (Python, capped) | tokens |
|---|---|---|---|
| [codeparrot-clean](https://huggingface.co/datasets/codeparrot/codeparrot-clean) | real `.py` files - fluency | 425k docs | 632M |
| [glaive-code-assistant](https://huggingface.co/datasets/glaiveai/glaive-code-assistant) | instruct | 122k | 44M |
| [TinyCodes](https://huggingface.co/datasets/nampdn-ai/tiny-codes) | instruct | 111k | 53M |
| [Magicoder-Evol-Instruct-110K](https://huggingface.co/datasets/ise-uiuc/Magicoder-Evol-Instruct-110K) | instruct | 106k | 44M |
| [CodeFeedback-Filtered](https://huggingface.co/datasets/m-a-p/CodeFeedback-Filtered-Instruction) | instruct | 88k | 57M |
| [Magicoder-OSS-Instruct-75K](https://huggingface.co/datasets/ise-uiuc/Magicoder-OSS-Instruct-75K) | instruct | 40k | 24M |
| [Cosmopedia v2](https://huggingface.co/datasets/HuggingFaceTB/smollm-corpus) | technical prose | 31M | 33.6B |

**A document mix is not a token mix.** Hugging Face's `interleave_datasets` samples by *document* probability, but the model learns from *tokens*. Mean document length varies about 4x across these sources, so a clean-looking 50/35/15 document split lands nowhere near 50/35/15 in tokens. To get the token mix we actually want, we have to solve for the document weights backwards.

## Part 4: Solving the mix backwards

For each source $s$, the filters (its length cap plus the language check) keep only a fraction $\sigma_s$ of the documents we stream - its **survival rate** - and the survivors have some mean length $L_s$ measured in *tokens*. (Token length is measured with the NanoCoder tokenizer; we build that tokenizer from scratch in the next notebook, so for now just take it as the thing that turns text into the integer tokens the model reads.)

If we sample source $s$ with document-probability $w_s$, its share of the **token** budget is:

$$\text{token\_share}_s \;=\; \frac{w_s \, L_s}{\sum_{j} w_j \, L_j}$$

We *want* a target token mix of roughly **78% code, 22% prose**. Rearranging the equation for $w_s$ given a target token share is the "backward solve" - it hands us the document weights that produce the token proportions we asked for.

The cell below runs the *forward* direction as a check: feed in the config's document weights and the measured $L_s$, and confirm the resulting token mix lands on target. The backward solve is the same equation, rearranged.

In [ ]:
import numpy as np
from nanocoder.data.config import DatasetConfig
dcfg = DatasetConfig()

# Which sources count toward the code side of the 78/22 target.
is_code = {"codeparrot": 1, "glaive": 1, "tinycodes": 1, "magicoder_evol": 1,
           "codefeedback": 1, "magicoder_oss": 1, "cosmo": 0}

names = list(dcfg.mix_proportions)
w = np.array([dcfg.mix_proportions[n] for n in names])
L = np.array([mean_tokens[n] for n in names])
tok_share = (w * L) / (w * L).sum()

code_share = sum(tok_share[i] for i, n in enumerate(names) if is_code[n])
print(f"{'source':<16}{'w_doc':>8}{'L_tok':>8}{'token %':>10}")
for i, n in enumerate(names):
    print(f"{n:<16}{w[i]:>8.3f}{L[i]:>8}{100*tok_share[i]:>9.1f}%")
print("-" * 42)
print(f"{'CODE tokens':<16}{'':>8}{'':>8}{100*code_share:>9.1f}%")
print(f"{'PROSE tokens':<16}{'':>8}{'':>8}{100*(1-code_share):>9.1f}%")
print("\nTarget was ~78% code / ~22% prose. Note codeparrot: 36% of the documents, 62% of"
      " the tokens.\nThat gap between the two columns is the whole point of Part 3.")

In [ ]:
# --- The measurement behind mean_tokens. Applies a source's filter and map in Python
# rather than on the stream, so the rejected docs stay visible.
def measure_source(raw_stream, keep, to_text, tokenizer, n=500):
    kept, total_tokens = 0, 0
    for ex in islice(raw_stream, n):
        txt = to_text(ex) if keep(ex) else None
        if txt:
            kept += 1
            total_tokens += len(tokenizer.encode(txt))
    return {"survival": kept / n, "mean_tokens": total_tokens / max(kept, 1)}

from huggingface_hub import snapshot_download
from nanocoder.tokenizer.tokenizer import NanoCoderTokenizer
tok = NanoCoderTokenizer.from_pretrained(snapshot_download("torq1/NanoCoder-tokenizer"))

print("glaive:", measure_source(
    load_dataset("glaiveai/glaive-code-assistant", split="train", streaming=True),
    keep=lambda ex: len(ex["answer"]) < dcfg.max_chars["glaive"],
    to_text=lambda ex: f"## Task\n{ex['question']}\n\n## Solution\n{ex['answer']}",
    tokenizer=tok))

# -- survival is the sigma_s in the Part 4 equation, mean_tokens is L_s. Run this per source
#    to rebuild the mean_tokens table above from scratch.

The compiled result:

| | tokens | share | epochs |
|---|---|---|---|
| raw Python | 1,264M | 57.4% | 2.0x |
| instruction (5 sets) | 443M | 20.2% | 2.0x |
| technical prose | 493M | 22.4% | 0.01x |
| **total** | **2,200M** | **78% code** | **~17.9 tok/param** |

codeparrot's cap sits at the tail of its length distribution rather than the median (89% of it survives): that distribution runs out to 361k characters, and files that long are machine-generated, not written - so the cap trims the generated tail without touching real code.

## Part 5: Token budget and epoch caps (how much data do we need?)

How much data is enough? The Chinchilla scaling result gives a usable rule of thumb, but to apply it we need to fix the model size early. The intuition is a balance:

- **Too many tokens per parameter** and we waste compute, and eventually risk memorizing rather than learning.
- **Too few tokens per parameter** and the model never sees enough to generalize in the first place.

Chinchilla's compute-optimal point sits near **~20 tokens per parameter**. For our ~123M-parameter base model, that lands right around **2.2B tokens** - which is where the mix in Part 4 was sized to.

This is why `max_samples` in `data/config.py` is **derived, not chosen**: it is the document count that lands the compiled corpus at ~2.2B tokens for our model size. The per-source character caps in `max_chars` do double duty - they bound memory *and* set each source's *epoch* count, since a small pool sampled often is simply seen more than once. Code sources are held to at most 2 epochs so the model generalizes across files rather than memorizing them.

Two sanity checks on the result:

- No individual code source is repeated more than 2x.
- The whole corpus stays inside the ~4-epoch band where repeating data still buys roughly what fresh data would (See *[Scaling Data-Constrained Language Models](https://arxiv.org/abs/2305.16264)*).

## Part 6: Cheap vs. expensive language filtering

We want Python, but the sources contain other languages. The rigorous way to check is to AST-parse every document - but at streaming scale, parsing millions of documents does not fit the throughput budget. So `looks_python` (in `data/sources.py`) is a deliberately *cheap* heuristic: it rejects on clear non-Python signals (`std::`, `public class`, a ```` ```rust ```` fence, ...) and accepts on Python signals (an explicit `'python'` tag, or the `def` / `import` co-occurrence).

Being cheap, it makes mistakes in both directions: it drops some valid Python (false negatives) and lets through the occasional non-Python document (false positives). That is the accepted price of a streaming pipeline - the parse-everything alternative is correct but far too slow. Where a source already ships a language column, the code trusts that column instead of the heuristic.

The honest gaps this leaves open - exact and near-duplicate removal, and their effect on memorization - are collected at the end of the notebook.

In [ ]:
# Used only where a source ships no language column.
non_py_kws = ['public class ', 'std::', 'printf(', 'fn main()', 'namespace ',
              'extern ', 'await fetch', 'System.out.print', 'using namespace std']
non_py_langs = ['javascript', 'js ', 'java ', 'cpp', 'c++', 'rust',
                'c#', 'csharp', 'ruby', 'swift', 'golang', 'sql']

def looks_python(text: str) -> bool:
    t = text.lower()
    if any(k in t for k in non_py_kws):              return False
    if any(f"```{l}" in t for l in non_py_langs):     return False
    return ('python' in t) or ('def ' in t and 'import ' in t)

# (snippet, is it actually Python?)
probes = [
    ("```python\ndef f(x): return x\n```",           True),
    ("import os\ndef main(): print(os.getcwd())",    True),
    ("def f(x):\n    return x * 2",                  True),
    ("```rust\nfn main() { }\n```",                  False),
    ("public class Main { }",                        False),
]
for text, is_py in probes:
    kept = looks_python(text)
    print(f"{'kept   ' if kept else 'dropped'} {text.splitlines()[0][:38]!r}"
          f"{'' if kept == is_py else '   <- heuristic is wrong here'}")

# -- The third probe is the accepted cost: real Python carrying neither a 'python' tag nor
#    an import is dropped. ast.parse would catch it and is far too slow at streaming scale.

## Part 7: Normalizing every source to one `text` field

The seven sources arrive with seven different schemas. Before they can be mixed, each has to expose a single `text` field. Three small mappers in `data/sources.py` do this:

- `as_task(ex, q, a)` builds `"## Task\n{q}\n\n## Solution\n{a}"` - the exact prompt shape the model will be asked to complete at inference.
- `as_fenced_py(ex)` wraps a raw file in a ```python fence.
- `as_cosmo(ex)` passes technical prose straight through.

**Why raw `.py` must be fenced:** the NanoCoder tokenizer only inserts its indentation markers *inside* a fenced code block. We build that tokenizer in the next notebook - for now the point is just that unfenced source code would skip the special handling every code block gets, so we fence raw files to route them through the same path as everything else.

In [ ]:
# src/nanocoder/data/sources.py

def as_task(ex, p_col, r_col):
    p, r = (ex.get(p_col) or '').strip(), (ex.get(r_col) or '').strip()
    return {"text": f"## Task\n{p}\n\n## Solution\n{r}" if (p and r) else None}

def as_fenced_py(ex):
    # Raw .py MUST be fenced: preprocess() only applies <|indent|>/<|dedent|> and
    # FIM inside a `' block, so unfenced source would silently bypass both.
    c = (ex.get('content') or '').strip()
    return {"text": f"```python\n{c}\n```" if c else None}

COSMO_FMT = ['textbook', 'textbook_unconditionned_topic', 'wikihow', 'textbook_narrative',
             'e-learning_module', 'textbook_academic', 'scientific_article']
def as_cosmo(ex):
    t = (ex.get('text') or '').strip()
    return {"text": t if t else None}

print(as_task({"question": "Reverse a string.", "answer": "def rev(s):\n    return s[::-1]"},
              "question", "answer")["text"])
print("-" * 60)
print(as_fenced_py({"content": "x = 1\nprint(x)"})["text"])

## Part 8: The train/validation split

`build_dataset` (in `data/sources.py`) routes **every `val_stride`-th valid document** into the validation set, where `val_stride = round(1 / val_split)`, capped at `max_samples * val_split`. Striding - rather than slicing off a random tail - keeps the validation set's *source proportions* matched to the training set, so val loss reflects the same mix the model trains on. The split is made at the **document** level.

**Honest caveat:** this split does *not* remove near-duplicates that straddle train and validation, so validation loss can read slightly optimistic. It is listed among the open gaps at the end of the notebook.

In [ ]:
# --- Compile the whole corpus. Hours of streaming; --max-samples on the CLI shrinks it.
import gc
from datasets import Dataset, DatasetDict, Features, Value
from nanocoder.data.sources import build_dataset

dcfg = DatasetConfig()
train_texts, val_texts = build_dataset(dcfg)

FEATURES = Features({"text": Value("large_string")})

# One split at a time, dropping each source list as soon as it is converted. Both the Python
# strings and their Arrow copy are resident during a conversion, and that is the memory peak
# of this whole notebook - use a High-RAM runtime.
ds_train = Dataset.from_dict({"text": train_texts}, features=FEATURES)
del train_texts; gc.collect()
ds_val = Dataset.from_dict({"text": val_texts}, features=FEATURES)
del val_texts; gc.collect()

ds = DatasetDict({ "train": ds_train, "validation": ds_val })
print(ds)

In [ ]:
# --- Checkpoint before uploading. The stream above is the expensive part of this notebook,
# so landing it on disk makes any later failure - a rejected upload, a dropped runtime - cost
# minutes instead of a full re-run.


## Part 9: Push to the Hugging Face Hub

With the corpus compiled and split, the last step is to publish it. Pushing to the Hub gives us a versioned, shareable dataset that the next notebooks load directly - the tokenizer trains on it, and the base model trains on the tokenized result.

Before uploading, save the corpus to disk (in case anything goes wrong)

In [ ]:
ds.save_to_disk("corpus_local")

# Resume from here after a restart, then skip straight to the push cell:
#   from datasets import load_from_disk
#   ds = load_from_disk("corpus_local")   # memory-mapped, not read into RAM
#
# Colab's local disk dies with the runtime. To survive a disconnect, mount Drive first and
# save under /content/drive/MyDrive/ instead - check the free quota, this is several GB.

In [ ]:
REPO_ID = "torq1/NanoCoder-pretrain"

ds.push_to_hub(REPO_ID, private=False)
print(f"Pushed to {REPO_ID} "
      f"({len(ds['train']):,} train / {len(ds['validation']):,} val docs)")

# -- Counts come off `ds`, not the source lists, so `del train_texts, val_texts` before
#    pushing is safe when memory is tight.
# -- Same run from a shell: python -m nanocoder.data.build_pretrain --repo-id <user>/NanoCoder-pretrain

## Next: the tokenizer

We now have a properly constructed, correctly-proportioned pretraining dataset. The next notebook uses it to train a **custom** byte-level BPE tokenizer built for Python-heavy text - the piece that turns all this text into the integers the model actually reads.

[NanoCoder 2: Tokenizer](https://colab.research.google.com/drive/1jKz9KB1qISMOmQn7xffbHVzP4WiNOXcu?usp=sharing)

---

## Appendix: open gaps not implemented here

These are known limitations of the pipeline above, listed honestly rather than hidden:

- **Benchmark decontamination.** Nothing here checks whether a training document overlaps with an evaluation set (HumanEval, MBPP, and the like). If eval problems leak into pretraining, the pass@k numbers reported in later notebooks are inflated and part of the apparent improvement is fictitious. This is the single most important gap to close before trusting any eval number - a substring or n-gram overlap filter against the eval prompts belongs right after the language filter.
- **Exact and near-duplicate removal.** There is no MinHash/LSH dedup pass. Duplicated code inflates memorization and makes validation loss optimistic (see Part 8).
- **Code-specific quality filtering.** No parseability check, license heuristics, or auto-generation heuristics beyond the length caps.